<a href="https://colab.research.google.com/github/priyanga2004113-coder/Smart-Agriculture-Analytics-/blob/main/Smart_Agriculture_Analytics1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Smart Agriculture Analytics
Dataset:Dataset: https://www.kaggle.com/datasets/sanchitagholap/crop-and-fertilizer-dataset-for-westernmaharashtra?utm_

In [1]:
import pandas as pd

df = pd.read_csv("/content/Crop and fertilizer dataset.csv")

df.head()

,District_Name,Soil_color,Nitrogen,Phosphorus,Potassium,pH,Rainfall,Temperature,Crop,Fertilizer,Link
0,Kolhapur,Black,75,50,100,6.5,1000,20,Sugarcane,Urea,https://youtu.be/2t5Am0xLTOo
1,Kolhapur,Black,80,50,100,6.5,1000,20,Sugarcane,Urea,https://youtu.be/2t5Am0xLTOo
2,Kolhapur,Black,85,50,100,6.5,1000,20,Sugarcane,Urea,https://youtu.be/2t5Am0xLTOo
3,Kolhapur,Black,90,50,100,6.5,1000,20,Sugarcane,Urea,https://youtu.be/2t5Am0xLTOo
4,Kolhapur,Black,95,50,100,6.5,1000,20,Sugarcane,Urea,https://youtu.be/2t5Am0xLTOo


In [2]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nSummary:")
display(df.describe())

Shape: (4513, 11)

Columns:
['District_Name', 'Soil_color', 'Nitrogen', 'Phosphorus', 'Potassium', 'pH', 'Rainfall', 'Temperature', 'Crop', 'Fertilizer', 'Link']

Data Types:
District_Name     object
Soil_color        object
Nitrogen           int64
Phosphorus         int64
Potassium          int64
pH               float64
Rainfall           int64
Temperature        int64
Crop              object
Fertilizer        object
Link              object
dtype: object

Summary:


,Nitrogen,Phosphorus,Potassium,pH,Rainfall,Temperature
count,4513.000000,4513.000000,4513.000000,4513.000000,4513.000000,4513.000000
mean,95.409927,54.341901,63.595170,6.715267,819.189010,25.915134
std,38.060648,16.551991,35.691911,0.625198,251.730813,5.897328
min,20.000000,10.000000,5.000000,5.500000,300.000000,10.000000
25%,60.000000,40.000000,40.000000,6.000000,600.000000,20.000000
50%,105.000000,55.000000,55.000000,6.500000,800.000000,25.000000
75%,125.000000,65.000000,75.000000,7.000000,1000.000000,30.000000
max,150.000000,90.000000,150.000000,8.500000,1700.000000,40.000000


In [ ]:
print("Missing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

Missing Values:
District_Name    0
Soil_color       0
Nitrogen         0
Phosphorus       0
Potassium        0
pH               0
Rainfall         0
Temperature      0
Crop             0
Fertilizer       0
Link             0
dtype: int64

Duplicate Rows: 0


In [ ]:
# Remove extra spaces from column names
df.columns = df.columns.str.strip()

# Remove extra spaces from text columns
text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

# Convert numerical columns
num_cols = [
    "Nitrogen",
    "Phosphorus",
    "Potassium",
    "pH",
    "Rainfall",
    "Temperature"
]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove invalid values
df.loc[df["pH"] < 0, "pH"] = None
df.loc[df["Rainfall"] < 0, "Rainfall"] = None
df.loc[df["Temperature"] < -50, "Temperature"] = None

print("Cleaning completed.")

Cleaning completed.


In [ ]:
# Detect outliers using IQR method

num_cols = [
    "Nitrogen",
    "Phosphorus",
    "Potassium",
    "pH",
    "Rainfall",
    "Temperature"
]

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()

    print(f"{col}: {outliers} outliers")

Nitrogen: 0 outliers
Phosphorus: 0 outliers
Potassium: 0 outliers
pH: 0 outliers
Rainfall: 0 outliers
Temperature: 0 outliers


In [ ]:
# Fill numerical missing values with median
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical missing values with mode
for col in text_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

# Remove duplicates
df = df.drop_duplicates()

print("Final Shape:", df.shape)
print("\nRemaining Missing Values:")
print(df.isnull().sum().sum())

Final Shape: (4513, 11)

Remaining Missing Values:
0


In [ ]:
import plotly.express as px

crop_counts = df["Crop"].value_counts().reset_index()
crop_counts.columns = ["Crop", "Count"]

fig = px.bar(
    crop_counts,
    x="Crop",
    y="Count",
    title="Crop Distribution",
    text="Count"
)

fig.show()

In [ ]:
import plotly.express as px

nutrient_avg = (
    df.groupby("Crop")[["Nitrogen", "Phosphorus", "Potassium"]]
      .mean()
      .reset_index()
)

nutrient_avg = nutrient_avg.melt(
    id_vars="Crop",
    var_name="Nutrient",
    value_name="Average_Value"
)

fig = px.bar(
    nutrient_avg,
    x="Crop",
    y="Average_Value",
    color="Nutrient",
    barmode="group",
    title="Average Soil Nutrient Levels Across Crops",
    labels={
        "Average_Value": "Average Nutrient Value",
        "Crop": "Crop"
    }
)

fig.show()

In [ ]:
import plotly.express as px

rainfall_avg = (
    df.groupby("Crop")["Rainfall"]
      .mean()
      .reset_index()
)

fig = px.bar(
    rainfall_avg,
    x="Crop",
    y="Rainfall",
    title="Average Rainfall Across Crops",
    text_auto=".1f"
)

fig.show()

In [ ]:
temperature_avg = (
    df.groupby("Crop")["Temperature"]
      .mean()
      .reset_index()
)

fig = px.bar(
    temperature_avg,
    x="Crop",
    y="Temperature",
    title="Average Temperature Across Crops",
    text_auto=".1f"
)

fig.show()

In [ ]:
fig = px.scatter(
    df,
    x="Rainfall",
    y="Temperature",
    color="Crop",
    hover_data=["District_Name", "Soil_color"],
    title="Environmental Conditions Suitable for Different Crops"
)

fig.show()

In [ ]:
region_crop = (
    df.groupby(["District_Name", "Crop"])
      .size()
      .reset_index(name="Count")
)

fig = px.bar(
    region_crop,
    x="District_Name",
    y="Count",
    color="Crop",
    title="Crop Distribution Across Districts"
)

fig.show()

In [ ]:
from sklearn.model_selection import train_test_split

features = [
    "District_Name",
    "Soil_color",
    "Nitrogen",
    "Phosphorus",
    "Potassium",
    "pH",
    "Rainfall",
    "Temperature"
]

X = df[features]
y = df["Crop"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
# Check duplicate feature rows
print("Duplicate feature rows:", df[features].duplicated().sum())
duplicate_rows = df[df[features].duplicated(keep=False)]

display(
    duplicate_rows.sort_values(features)
)
df = df.drop_duplicates(subset=features).reset_index(drop=True)

print("Duplicate feature rows after removal:",
      df[features].duplicated().sum())

print("New shape:", df.shape)

Duplicate feature rows: 3


,District_Name,Soil_color,Nitrogen,Phosphorus,Potassium,pH,Rainfall,Temperature,Crop,Fertilizer,Link
788,Kolhapur,Dark Brown,100,40,40.0,6.0,500.0,15.0,Wheat,Urea,https://youtu.be/VNptnPJt5yo
921,Kolhapur,Dark Brown,100,40,40.0,6.0,500.0,15.0,Wheat,19:19:19 NPK,https://youtu.be/VNptnPJt5yo
793,Kolhapur,Dark Brown,100,45,45.0,6.5,600.0,20.0,Wheat,Urea,https://youtu.be/VNptnPJt5yo
930,Kolhapur,Dark Brown,100,45,45.0,6.5,600.0,20.0,Wheat,19:19:19 NPK,https://youtu.be/VNptnPJt5yo
3889,Sangli,Black,25,20,40.0,5.5,1500.0,20.0,Turmeric,Urea,https://youtu.be/RiBSAjxgFvU
3910,Sangli,Black,25,20,40.0,5.5,1500.0,20.0,Turmeric,White Potash,https://youtu.be/RiBSAjxgFvU


Duplicate feature rows after removal: 0
New shape: (4510, 11)


In [ ]:
X = df[features]
y = df["Crop"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

categorical_cols = ["District_Name", "Soil_color"]

numerical_cols = [
    "Nitrogen", "Phosphorus", "Potassium",
    "pH", "Rainfall", "Temperature"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numerical_cols)
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9988913525498891

Classification Report:
              precision    recall  f1-score   support

      Cotton       1.00      1.00      1.00       130
      Ginger       1.00      1.00      1.00        25
        Gram       1.00      1.00      1.00        16
      Grapes       1.00      1.00      1.00        25
   Groundnut       1.00      1.00      1.00        35
       Jowar       1.00      1.00      1.00        79
       Maize       0.99      1.00      0.99        70
      Masoor       1.00      1.00      1.00         2
       Moong       1.00      1.00      1.00        20
        Rice       1.00      1.00      1.00        62
     Soybean       1.00      1.00      1.00         9
   Sugarcane       1.00      1.00      1.00       202
         Tur       1.00      1.00      1.00        25
    Turmeric       1.00      1.00      1.00        11
        Urad       1.00      1.00      1.00        20
       Wheat       1.00      0.99      1.00       171

    accuracy               

In [ ]:
from sklearn.metrics import confusion_matrix
import plotly.express as px

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=model.classes_
)

fig = px.imshow(
    cm,
    x=model.classes_,
    y=model.classes_,
    text_auto=True,
    title="Crop Prediction Confusion Matrix",
    labels={
        "x": "Predicted Crop",
        "y": "Actual Crop"
    }
)

fig.show()

In [ ]:
def recommend_crop(
    district,
    soil,
    nitrogen,
    phosphorus,
    potassium,
    ph,
    rainfall,
    temperature,
    top_n=3
):
    input_data = pd.DataFrame([{
        "District_Name": district,
        "Soil_color": soil,
        "Nitrogen": nitrogen,
        "Phosphorus": phosphorus,
        "Potassium": potassium,
        "pH": ph,
        "Rainfall": rainfall,
        "Temperature": temperature
    }])

    probabilities = model.predict_proba(input_data)[0]
    crops = model.classes_

    result = pd.DataFrame({
        "Crop": crops,
        "Suitability_Score": probabilities * 100
    })

    result = result.sort_values(
        "Suitability_Score",
        ascending=False
    ).head(top_n)

    return result.reset_index(drop=True)

In [ ]:
recommend_crop(
    district="Thanjavur",
    soil="Black",
    nitrogen=80,
    phosphorus=40,
    potassium=40,
    ph=6.5,
    rainfall=100,
    temperature=28
)

,Crop,Suitability_Score
0,Wheat,23.666667
1,Jowar,22.333333
2,Urad,13.666667


In [ ]:
df.to_csv(
    "/content/Smart_Agriculture_Cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
